In [ ]:
from model.model import load_model
from dataset.dataset import load_data, create_dataloaders
from functions.optimizer import load_optimizer
from functions.loss import load_loss_fun
from functions.functions import train_model, eval_model
from functions.wb_eval import wb_train
from functions.xai import explain_dataset, evaluate_explainations, visualize_k_expl
from functions.loss import load_loss_fun
from experiments.utils import enable_reproducibility
import torch

In [ ]:
seed=123
loss_name= "CrossEntrpy"

use_cuda = torch.cuda.is_available()
device = 'cuda' if use_cuda else 'cpu'
enable_reproducibility(seed)

model = load_model("ResNet", model_name="resnet50", n_classes=2, pretrained=True, device=device)
optim = load_optimizer("SGD", model.parameters(), lr=lr)
if loss_name == "CrossEntropy":
  loss = load_loss_fun(loss_name)
else:
  loss = load_loss_fun("RRR", reg_rate = 1e2, normalize=True)

train_set, val_set, test_set = load_data(
  "Waterbirds", reload=False, balance=True, seed=seed)
data = [train_set, val_set, test_set]
params = {"batch_size":64}
m_params = [params]*3
train_loader, val_loader, test_loader = create_dataloaders(data, m_params)

log, dyn = wb_train(
  model=model, 
  train_loader=train_loader, 
  optimizer=optim, 
  loss_fun=loss, 
  n_epochs=100, 
  eval_loader=val_loader, 
  patience=3,
  device=device
)

all_attr, all_imgs = explain_dataset(train_loader, model, device)
exp_penalty, class_penalty = evaluate_explainations(all_attr, train_set.masks, train_set.y)
print(exp_penalty)
print(class_penalty)

visualize_k_expl(all_attr, all_imgs, train_set, 0, k=3)

In [ ]:
visualize_k_expl(all_attr, all_imgs, train_set, 0, k=3)